# 17 — V5 Role-Only Expansion → Final DEV/CONFIRM Freeze

The prior V5 roster attempt correctly stopped because the existing role-only seed universe had only:

- Competitive 45 fresh AA
- Project 50 fresh AA
- Status 14 fresh AA

This notebook does **not** lower the target and does **not** reuse prior subjects.

Instead it:

1. Loads the already frozen Wave A available pool.
2. Queries Wikidata using **occupation only** to generate a larger role-only candidate universe.
3. Resolves those names against the **same frozen VedAstro/Rodden-AA birth snapshot**.
4. Excludes all consumed subjects and all names already present in the original V4 role seed.
5. Combines new eligible candidates with Wave A.
6. Deterministically freezes:
   - DEV 160
   - CONFIRM 80
7. Creates an event-intake template for **DEV only**.

No events, chronology, pairability, astrology, or Control scores are read.


In [1]:

from pathlib import Path
from datetime import datetime
import ast
import hashlib
import json
import re
import time
import unicodedata
import urllib.parse
import urllib.request
import urllib.error

import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_VERSION = "SAJU_ML_V5_ROLE_ONLY_EXPANSION_FINAL_FREEZE_20260817"
SELECTION_SEED = 2026081701

DEV_TARGET = {"COMPETITIVE":40, "PROJECT":50, "STATUS":70}
CONFIRM_TARGET = {"COMPETITIVE":20, "PROJECT":25, "STATUS":35}

FEMALE_MIN_SHARE = 0.20
BIRTH_YEAR_MIN = 1900
BIRTH_YEAR_MAX = 1995

WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
MIN_SITELINKS = 5
PER_ROLE_LIMIT = 5000

ROLE_QIDS = {
    "COMPETITIVE": {
        "athlete": "Q2066131",
        "association_football_player": "Q937857",
        "basketball_player": "Q3665646",
        "tennis_player": "Q10833314",
        "boxer": "Q11338576",
    },
    "PROJECT": {
        "actor": "Q33999",
        "film_director": "Q2526255",
        "musician": "Q639669",
        "singer": "Q177220",
    },
    "STATUS": {
        "politician": "Q82955",
        "businessperson": "Q43845",
    },
}

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p] + list(p.parents):
        if (c / "saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside the Chartpalja saju repository.")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def norm_name(x):
    s = unicodedata.normalize("NFKD", str(x))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.casefold()
    return re.sub(r"[^a-z0-9]+", "", s)

def is_female(x):
    return str(x).strip().lower().startswith("female")

def det_key(name, axis, split):
    raw = "%s|%s|%s|%s" % (
        norm_name(name), axis, split, SELECTION_SEED
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

ROOT = find_repo_root()

V5_OUT = ROOT / "research/ml/artifacts/v5_roster_freeze"
WAVE_A_PATH = V5_OUT / "V5_ROSTER_WAVE_A_AVAILABLE_POOL.csv"
EXP_REQ_PATH = V5_OUT / "V5_ROSTER_EXPANSION_REQUEST.json"

V4_W1_DIR = ROOT / "research/ml/artifacts/v4_unified_dev_roster"
V4_W2_DIR = ROOT / "research/ml/artifacts/v4_unified_dev_wave2_roster"
V4_SEED_DIR = ROOT / "research/ml_corpus/v4_unified_dev_wave2"

BIRTH_PATH = V4_W1_DIR / "PersonList-15k.csv"
ORIGINAL_SEED_PATH = V4_SEED_DIR / "V4_UNIFIED_DEV_WAVE2_PREDECLARED_SEED_UNIVERSE_R3.csv"
W1_PATH = V4_W1_DIR / "V4_UNIFIED_DEV_SUBJECT_ROSTER_100.csv"
W2_PATH = V4_W2_DIR / "V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_44.csv"

SPEC_PATH = ROOT / "research/ml_corpus/v5_ground_truth/V5_ROLE_ONLY_WIKIDATA_EXPANSION_SPEC.json"

ART = ROOT / "research/ml/artifacts/v5_final_roster"
ART.mkdir(parents=True, exist_ok=True)

QUERY_CACHE = ART / "wikidata_role_query_cache"
QUERY_CACHE.mkdir(parents=True, exist_ok=True)

for p in [
    WAVE_A_PATH, EXP_REQ_PATH, BIRTH_PATH, ORIGINAL_SEED_PATH,
    W1_PATH, W2_PATH, SPEC_PATH
]:
    if not p.exists():
        raise FileNotFoundError(p)

wave_a = pd.read_csv(WAVE_A_PATH)
exp_req = json.load(open(EXP_REQ_PATH, encoding="utf-8"))
birth = pd.read_csv(BIRTH_PATH)
original_seed = pd.read_csv(ORIGINAL_SEED_PATH)
w1 = pd.read_csv(W1_PATH)
w2 = pd.read_csv(W2_PATH)
spec = json.load(open(SPEC_PATH, encoding="utf-8"))

assert exp_req["status"] == (
    "V5_CURRENT_SEED_UNIVERSE_INSUFFICIENT_"
    "EXPANSION_REQUIRED_BEFORE_FINAL_ROSTER_FREEZE"
)
assert sha256_file(WAVE_A_PATH) == exp_req["wave_a_pool_sha256"]
assert len(wave_a) == 109
assert wave_a.groupby("axis").size().to_dict() == {
    "COMPETITIVE":45, "PROJECT":50, "STATUS":14
}
assert spec["status"] == "PREDECLARED_BEFORE_ROLE_EXPANSION_QUERY"

print("Preflight PASS")
print("Wave A:", wave_a.groupby("axis").size().to_dict())
print("Birth snapshot SHA:", sha256_file(BIRTH_PATH))


Preflight PASS
Wave A: {'COMPETITIVE': 45, 'PROJECT': 50, 'STATUS': 14}
Birth snapshot SHA: ca28a3fea1250b2ea01eb06a54b4921ec0bf790fdc14c5c561006f1c592c634b


## 1. Build the full consumed-name exclusion set

In [2]:

def names_from_json(path):
    if not path.exists():
        return []
    try:
        obj = json.load(open(path, encoding="utf-8"))
    except Exception:
        return []
    return [
        s.get("name")
        for s in obj.get("subjects", [])
        if s.get("name")
    ]

prior_paths = {
    "V1": ROOT / "research/ml_corpus/v1/SAJU_ML_CORPUS_V1.json",
    "V2_NEW_DEV": ROOT / "research/ml_corpus/v2/SAJU_ML_CORPUS_V2_NEW_DEV.json",
    "NEW_DEV_2": ROOT / "research/ml_corpus/new_dev_2/SAJU_ML_NEW_DEV_2_CORPUS.json",
    "V4_TARGET_EXPANSION": ROOT / "research/ml_corpus/v4_target_expansion/V4_TARGET_EXPANSION_CORPUS.json",
}

consumed_names = []
for label, path in prior_paths.items():
    ns = names_from_json(path)
    consumed_names.extend(ns)
    print(label, len(ns))

consumed_norm = {norm_name(x) for x in consumed_names}
consumed_norm |= set(w1["name"].map(norm_name))
consumed_norm |= set(w2["name"].map(norm_name))

# Expansion must truly be outside the original 915-name role seed universe.
original_seed_norm = set(original_seed["name"].map(norm_name))
wave_a_norm = set(wave_a["name"].map(norm_name))

# Wave A itself is allowed even though it is part of original_seed.
assert wave_a_norm.issubset(original_seed_norm)

for p in prior_paths.values():
    u = str(p).upper()
    assert "NEW_CONFIRM" not in u
    assert "VALIDATION_B" not in u
    assert "PUBLIC_CHECK" not in u
    assert "PUBLIC_FINAL" not in u

print("Consumed unique normalized names:", len(consumed_norm))
print("Original role-seed names blocked from expansion:", len(original_seed_norm))


V1 94
V2_NEW_DEV 70
NEW_DEV_2 100
V4_TARGET_EXPANSION 169
Consumed unique normalized names: 376
Original role-seed names blocked from expansion: 915


## 2. Parse the frozen VedAstro snapshot and enforce Rodden AA

In [3]:

required_birth_cols = {"RowKey","BirthTime","Gender","Name","Notes"}
if not required_birth_cols.issubset(set(birth.columns)):
    raise RuntimeError(
        "Unexpected PersonList-15k schema. Found: %s"
        % list(birth.columns)
    )

def parse_birth_blob(blob):
    obj = json.loads(str(blob))
    std = obj["StdTime"]
    loc = obj["Location"]

    m = re.match(
        r"^(\d{1,2}):(\d{2})\s+(\d{2})/(\d{2})/(\d{4})\s+([+-]\d{2}:\d{2})$",
        std.strip()
    )
    if not m:
        raise ValueError(std)

    hh, mm, dd, mo, yyyy, offset = m.groups()

    return {
        "birth_date":"%04d-%02d-%02d" % (int(yyyy),int(mo),int(dd)),
        "birth_time":"%02d:%02d" % (int(hh),int(mm)),
        "utc_offset":offset,
        "birth_year":int(yyyy),
        "birth_place":loc["Name"],
        "longitude":float(loc["Longitude"]),
        "latitude":float(loc["Latitude"]),
    }

def rodden_grade(notes):
    s = str(notes)
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, dict):
            return str(obj.get("rodden") or "").strip().upper()
    except Exception:
        pass

    m = re.search(
        r"rodden['\"]?\s*:\s*['\"]([^'\"]+)['\"]",
        s,
        flags=re.I
    )
    return m.group(1).strip().upper() if m else ""

birth_rows = []
grade_counts = {}

for _, r in birth.iterrows():
    grade = rodden_grade(r["Notes"])
    grade_counts[grade] = grade_counts.get(grade, 0) + 1

    if grade != "AA":
        continue

    try:
        p = parse_birth_blob(r["BirthTime"])
    except Exception:
        continue

    birth_rows.append({
        "source_row_key": r["RowKey"],
        "source_name": r["Name"],
        "gender": str(r["Gender"]).lower(),
        "norm_name": norm_name(r["Name"]),
        **p,
    })

aa = pd.DataFrame(birth_rows)

name_counts = aa.groupby("norm_name").size()
ambiguous = set(name_counts[name_counts > 1].index)

aa_unique = aa[
    ~aa.norm_name.isin(ambiguous)
].copy()

aa_unique = aa_unique[
    aa_unique.birth_year.between(
        BIRTH_YEAR_MIN, BIRTH_YEAR_MAX, inclusive="both"
    )
].copy()

print("Rodden grades:", grade_counts)
print("Unique AA names in birth-year window:", len(aa_unique))
print("Ambiguous AA normalized names excluded:", len(ambiguous))

if len(aa_unique) < 500:
    raise RuntimeError("Unexpectedly small strict-AA birth pool.")


Rodden grades: {'AA': 15790, 'FF': 17}
Unique AA names in birth-year window: 12785
Ambiguous AA normalized names excluded: 27


## 3. Query Wikidata occupation snapshots

Only these fields are requested:

- occupation identity
- English person label
- sitelink count (notability filter/ranking)

No events or career outcomes are requested.

Each role response is cached so a notebook rerun does not repeatedly hit WDQS.


In [5]:
# 3. Query Wikidata occupation snapshots
#
# Robust / lightweight WDQS version.
#
# Key changes:
# - Remove wikibase:sitelinks lookup
# - Remove ORDER BY
# - Use only occupation + English label
# - Reuse successful caches
# - A single failed role does NOT abort the whole notebook
# - Cell 5 will later decide whether the total candidate pool is sufficient

def wikidata_role_query(qid, limit=PER_ROLE_LIMIT):
    return """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?person ?personLabel WHERE {
  ?person wdt:P106 wd:%s ;
          rdfs:label ?personLabel .

  FILTER(LANG(?personLabel) = "en")
}
LIMIT %d
""" % (
        qid,
        int(limit),
    )


def run_wdqs_light(
    query,
    cache_path,
    max_attempts=3,
):
    # Reuse previously completed query cache.
    if cache_path.exists():
        print("  cache HIT")
        return json.load(
            open(
                cache_path,
                encoding="utf-8"
            )
        )

    params = urllib.parse.urlencode({
        "query": query,
        "format": "json",
    })

    url = (
        WIKIDATA_ENDPOINT
        + "?"
        + params
    )

    headers = {
        "Accept": (
            "application/"
            "sparql-results+json"
        ),
        "User-Agent": (
            "Chartpalja-Saju-Research/1.0 "
            "(role-only roster construction; "
            "no event scraping)"
        ),
    }

    last_error = None

    for attempt in range(max_attempts):
        try:
            req = urllib.request.Request(
                url,
                headers=headers,
            )

            with urllib.request.urlopen(
                req,
                timeout=75,
            ) as resp:
                obj = json.loads(
                    resp.read().decode("utf-8")
                )

            with open(
                cache_path,
                "w",
                encoding="utf-8",
            ) as f:
                json.dump(
                    obj,
                    f,
                    ensure_ascii=False,
                )

            return obj

        except (
            urllib.error.HTTPError,
            urllib.error.URLError,
            TimeoutError,
        ) as e:

            last_error = e

            print(
                "  attempt",
                attempt + 1,
                "failed:",
                repr(e),
            )

            if attempt + 1 < max_attempts:
                # Small backoff.
                # Query itself is now much cheaper,
                # so long waits should not be necessary.
                sleep_s = 5 * (
                    attempt + 1
                )

                print(
                    "  retry in",
                    sleep_s,
                    "sec"
                )

                time.sleep(sleep_s)

        except Exception as e:
            last_error = e

            print(
                "  unexpected failure:",
                repr(e)
            )

            break

    # IMPORTANT:
    # Do not kill the entire research notebook
    # because one external occupation query failed.
    return {
        "_failed": True,
        "_error": repr(last_error),
        "results": {
            "bindings": []
        }
    }


role_rows = []
query_failures = []


for axis, role_map in ROLE_QIDS.items():

    for role_name, qid in role_map.items():

        # Use a NEW cache suffix because the query
        # definition changed from the old expensive version.
        cache_path = (
            QUERY_CACHE
            / (
                "%s__%s__LIGHT_V1.json"
                % (
                    axis,
                    role_name,
                )
            )
        )

        print(
            "query:",
            axis,
            role_name,
            qid,
        )

        obj = run_wdqs_light(
            wikidata_role_query(
                qid,
                limit=PER_ROLE_LIMIT,
            ),
            cache_path,
        )

        if obj.get("_failed"):

            query_failures.append({
                "axis": axis,
                "role_family": role_name,
                "role_qid": qid,
                "error": obj.get(
                    "_error"
                ),
            })

            print(
                "  SKIPPED after retries"
            )

            continue


        bindings = (
            obj.get(
                "results",
                {}
            )
            .get(
                "bindings",
                []
            )
        )

        print(
            "  returned:",
            len(bindings)
        )


        for b in bindings:

            person_uri = (
                b.get("person")
                or {}
            ).get(
                "value",
                ""
            )

            label = (
                b.get("personLabel")
                or {}
            ).get(
                "value",
                ""
            )

            if (
                not person_uri
                or not label
            ):
                continue


            role_rows.append({
                "axis": axis,

                "role_family":
                    role_name,

                "role_qid":
                    qid,

                "wikidata_id":
                    person_uri.rsplit(
                        "/",
                        1
                    )[-1],

                "name":
                    label,

                "norm_name":
                    norm_name(
                        label
                    ),

                # Sitelink ranking is deliberately
                # removed from V1 lightweight query.
                "wikidata_sitelinks":
                    np.nan,

                "seed_origin":
                    (
                        "V5_WIKIDATA_"
                        "ROLE_ONLY_EXPANSION"
                    ),

                "selection_information_used":
                    (
                        "ROLE_CATEGORY_ONLY;"
                        "IDENTITY_LABEL;"
                        "NO_EVENT_OUTCOME;"
                        "NO_CHRONOLOGY;"
                        "NO_PAIRABILITY;"
                        "NO_ASTROLOGY;"
                        "NO_CONTROL"
                    ),
            })


if not role_rows:
    raise RuntimeError(
        "All Wikidata occupation "
        "queries failed. "
        "Do not continue."
    )


role_df = pd.DataFrame(
    role_rows
)


# --------------------------------------------------
# Deduplicate within the SAME axis.
#
# We no longer rank by sitelinks.
# Use deterministic role_family / Wikidata ID order.
# --------------------------------------------------

role_df = (
    role_df
    .sort_values(
        [
            "axis",
            "norm_name",
            "role_family",
            "wikidata_id",
        ]
    )
    .drop_duplicates(
        [
            "axis",
            "norm_name",
        ]
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------
# A person appearing across multiple study axes
# is excluded rather than manually classified.
# --------------------------------------------------

axis_n = (
    role_df
    .groupby(
        "norm_name"
    )["axis"]
    .nunique()
)

cross_axis_ambiguous = set(
    axis_n[
        axis_n > 1
    ].index
)


role_df = role_df[
    ~role_df[
        "norm_name"
    ].isin(
        cross_axis_ambiguous
    )
].copy()


# --------------------------------------------------
# Save immutable role-only query snapshot.
# --------------------------------------------------

role_snapshot_path = (
    ART
    / "V5_WIKIDATA_ROLE_ONLY_QUERY_SNAPSHOT.csv"
)

role_df.to_csv(
    role_snapshot_path,
    index=False,
)


# Query failures are diagnostic only.
failure_path = (
    ART
    / "V5_WIKIDATA_ROLE_QUERY_FAILURES.json"
)

with open(
    failure_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "n_failures":
                len(
                    query_failures
                ),

            "failures":
                query_failures,

            "policy":
                (
                    "Individual external "
                    "role-query failure "
                    "does not alter "
                    "membership rules. "
                    "Final combined-pool "
                    "sufficiency gate "
                    "decides continuation."
                ),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )


display(
    role_df
    .groupby(
        [
            "axis",
            "role_family",
        ]
    )
    .size()
    .rename("n")
    .reset_index()
)


print()
print(
    "Unique role-only candidates:",
    len(role_df)
)

print(
    "Cross-axis ambiguous names excluded:",
    len(
        cross_axis_ambiguous
    )
)

print(
    "Role queries skipped:",
    len(
        query_failures
    )
)

if query_failures:
    display(
        pd.DataFrame(
            query_failures
        )
    )

query: COMPETITIVE athlete Q2066131
  returned: 5000
query: COMPETITIVE association_football_player Q937857
  unexpected failure: JSONDecodeError('Expecting property name enclosed in double quotes: line 43314 column 1 (char 1047023)')
  SKIPPED after retries
query: COMPETITIVE basketball_player Q3665646
  attempt 1 failed: <HTTPError 502: 'Bad Gateway'>
  retry in 5 sec
  unexpected failure: JSONDecodeError('Unterminated string starting at: line 10861 column 7 (char 261935)')
  SKIPPED after retries
query: COMPETITIVE tennis_player Q10833314
  returned: 5000
query: COMPETITIVE boxer Q11338576
  attempt 1 failed: <HTTPError 502: 'Bad Gateway'>
  retry in 5 sec
  returned: 5000
query: PROJECT actor Q33999
  returned: 5000
query: PROJECT film_director Q2526255
  returned: 5000
query: PROJECT musician Q639669
  returned: 5000
query: PROJECT singer Q177220
  returned: 5000
query: STATUS politician Q82955
  returned: 5000
query: STATUS businessperson Q43845
  returned: 5000


,axis,role_family,n
0,COMPETITIVE,athlete,4919
1,COMPETITIVE,boxer,4903
2,COMPETITIVE,tennis_player,4911
3,PROJECT,actor,4876
4,PROJECT,film_director,3976
5,PROJECT,musician,3949
6,PROJECT,singer,2357
7,STATUS,businessperson,4847
8,STATUS,politician,4814



Unique role-only candidates: 39552
Cross-axis ambiguous names excluded: 325
Role queries skipped: 2


,axis,role_family,role_qid,error
0,COMPETITIVE,association_football_player,Q937857,JSONDecodeError('Expecting property name enclo...
1,COMPETITIVE,basketball_player,Q3665646,JSONDecodeError('Unterminated string starting ...


## 4. Resolve only genuinely new role candidates against strict-AA births

In [6]:

# Expansion candidates must be outside BOTH:
# - consumed names
# - original V4 915-name role seed
expansion = role_df[
    ~role_df.norm_name.isin(consumed_norm)
    & ~role_df.norm_name.isin(original_seed_norm)
].copy()

resolved_new = expansion.merge(
    aa_unique,
    on="norm_name",
    how="inner",
    suffixes=("","_birth")
)

# The Wikidata English label becomes the study name; source_name preserves
# the exact birth-snapshot spelling.
resolved_new["eligible"] = True
resolved_new["prior_used"] = False
resolved_new["birth_ok"] = True
resolved_new["wave"] = "V5_ROLE_ONLY_EXPANSION"
resolved_new["event_collection_started"] = False
resolved_new["astrology_scored"] = False

# Deduplicate any unexpected birth-row collision by construction.
assert resolved_new.groupby(["axis","norm_name"]).size().max() == 1

new_counts = resolved_new.groupby("axis").size().to_dict()

display(
    resolved_new.groupby(["axis","role_family"])
    .size().rename("strict_AA_matches").reset_index()
)

print("New strict-AA expansion counts:", new_counts)

resolved_new_path = ART / "V5_ROLE_ONLY_EXPANSION_STRICT_AA_POOL.csv"
resolved_new.to_csv(resolved_new_path, index=False)

# Audit against requested expansion deficits.
need = exp_req["DEV_plus_CONFIRM_expansion_deficit"]
for axis, deficit in need.items():
    got = int(new_counts.get(axis, 0))
    print(axis, "need at least", deficit, "new strict-AA; got", got)


,axis,role_family,strict_AA_matches
0,COMPETITIVE,athlete,58
1,COMPETITIVE,boxer,72
2,COMPETITIVE,tennis_player,56
3,PROJECT,actor,265
4,PROJECT,film_director,314
5,PROJECT,musician,218
6,PROJECT,singer,153
7,STATUS,businessperson,91
8,STATUS,politician,63


New strict-AA expansion counts: {'COMPETITIVE': 186, 'PROJECT': 950, 'STATUS': 154}
COMPETITIVE need at least 15 new strict-AA; got 186
PROJECT need at least 25 new strict-AA; got 950
STATUS need at least 91 new strict-AA; got 154


## 5. Combined-pool sufficiency gate

In [7]:

# Normalize Wave A schema to the columns needed for final deterministic selection.
wa = wave_a.copy()

for c in [
    "role_family","role_qid","wikidata_id","wikidata_sitelinks"
]:
    if c not in wa.columns:
        wa[c] = np.nan

wa["candidate_source"] = "V5_WAVE_A"
resolved_new["candidate_source"] = "V5_WIKIDATA_EXPANSION"

common_cols = sorted(
    set(wa.columns).intersection(set(resolved_new.columns))
)

combined = pd.concat(
    [wa[common_cols], resolved_new[common_cols]],
    ignore_index=True,
    sort=False
)

# No normalized name can occur twice in an axis after union.
combined = (
    combined
    .sort_values(["axis","norm_name","candidate_source"])
    .drop_duplicates(["axis","norm_name"])
    .reset_index(drop=True)
)

# Cross-axis duplicate identities are forbidden.
axis_n2 = combined.groupby("norm_name")["axis"].nunique()
bad_cross = set(axis_n2[axis_n2 > 1].index)
if bad_cross:
    combined = combined[
        ~combined.norm_name.isin(bad_cross)
    ].copy()

combined_counts = combined.groupby("axis").size().to_dict()

suff_rows = []
sufficient = True

for axis in DEV_TARGET:
    required = DEV_TARGET[axis] + CONFIRM_TARGET[axis]
    available = int(combined_counts.get(axis,0))
    ok = available >= required
    sufficient &= ok
    suff_rows.append({
        "axis":axis,
        "Wave_A":int((combined.candidate_source=="V5_WAVE_A").mul(combined.axis==axis).sum()),
        "new_expansion":int((combined.candidate_source=="V5_WIKIDATA_EXPANSION").mul(combined.axis==axis).sum()),
        "combined_available":available,
        "required_DEV_plus_CONFIRM":required,
        "sufficient":ok,
    })

suff = pd.DataFrame(suff_rows)
display(suff)

combined.to_csv(
    ART / "V5_COMBINED_ROLE_ONLY_STRICT_AA_POOL.csv",
    index=False
)

if not sufficient:
    diag = {
        "version":"V5_ROLE_EXPANSION_INSUFFICIENT_V1",
        "created_at":datetime.now().isoformat(timespec="seconds"),
        "status":"V5_ROLE_EXPANSION_STILL_INSUFFICIENT_DO_NOT_FREEZE",
        "counts":suff.to_dict(orient="records"),
        "event_collection_allowed":False,
        "astrology_generation_allowed":False,
        "rule":"Do not lower targets or reuse prior subjects."
    }
    json.dump(
        diag,
        open(ART/"V5_ROLE_EXPANSION_INSUFFICIENT.json","w",encoding="utf-8"),
        ensure_ascii=False,
        indent=2
    )
    raise RuntimeError(
        "Role-only expansion still insufficient. "
        "Send V5_ROLE_EXPANSION_INSUFFICIENT.json."
    )

print("Combined sufficiency PASS")


,axis,Wave_A,new_expansion,combined_available,required_DEV_plus_CONFIRM,sufficient
0,COMPETITIVE,45,186,231,60,True
1,PROJECT,50,950,1000,75,True
2,STATUS,14,154,168,105,True


Combined sufficiency PASS


## 6. Deterministically freeze DEV, then sealed CONFIRM

In [8]:

def choose(pool, target, split):
    selected = []

    for axis, n in target.items():
        g = pool[pool.axis == axis].copy()
        g["final_det_key"] = g.apply(
            lambda r: det_key(r["name"], axis, split),
            axis=1
        )
        g = g.sort_values(
            ["final_det_key","norm_name"]
        ).reset_index(drop=True)

        # Frozen female guardrail. It never uses event information.
        females = g[g.gender.map(is_female)].copy()
        min_female = int(np.ceil(n * FEMALE_MIN_SHARE))
        take_f = min(min_female, len(females))

        chosen_f = females.head(take_f)
        remaining = g[
            ~g.norm_name.isin(set(chosen_f.norm_name))
        ].copy()

        chosen = pd.concat(
            [chosen_f, remaining.head(n - take_f)],
            ignore_index=True
        )

        if len(chosen) != n:
            raise RuntimeError(
                "%s %s final selection %d/%d"
                % (split, axis, len(chosen), n)
            )

        selected.append(chosen)

    return pd.concat(selected, ignore_index=True)

dev = choose(combined, DEV_TARGET, "DEV")

confirm_pool = combined[
    ~combined.norm_name.isin(set(dev.norm_name))
].copy()

confirm = choose(confirm_pool, CONFIRM_TARGET, "CONFIRM")

assert len(dev) == 160
assert len(confirm) == 80
assert set(dev.norm_name).isdisjoint(set(confirm.norm_name))
assert not set(dev.norm_name) & consumed_norm
assert not set(confirm.norm_name) & consumed_norm

def finalize(df, split):
    x = df.copy().reset_index(drop=True)
    x["subject_id"] = [
        "V5%s_%03d" % (split, i+1)
        for i in range(len(x))
    ]
    x["preassigned_axis"] = x["axis"]
    x["rodden_rating"] = "AA"
    x["birth_source"] = "Frozen VedAstro PersonList-15k snapshot"
    x["split"] = split
    x["event_collection_started"] = False
    x["astrology_scored"] = False

    cols = [
        "subject_id","name","source_name","gender","preassigned_axis",
        "birth_date","birth_time","utc_offset","birth_place","latitude","longitude",
        "rodden_rating","source_row_key","birth_source",
        "candidate_source","role_family","role_qid","wikidata_id","wikidata_sitelinks",
        "seed_origin","selection_information_used",
        "event_collection_started","astrology_scored","split","final_det_key"
    ]
    return x[[c for c in cols if c in x.columns]]

dev = finalize(dev, "DEV")
confirm = finalize(confirm, "CONFIRM")

assert dev.preassigned_axis.value_counts().to_dict() == {
    "STATUS":70,"PROJECT":50,"COMPETITIVE":40
}
assert confirm.preassigned_axis.value_counts().to_dict() == {
    "STATUS":35,"PROJECT":25,"COMPETITIVE":20
}

print("DEV composition")
display(
    dev.groupby(["preassigned_axis","candidate_source"])
    .size().rename("n").reset_index()
)

print("CONFIRM composition")
display(
    confirm.groupby(["preassigned_axis","candidate_source"])
    .size().rename("n").reset_index()
)

print("DEV gender")
display(
    dev.groupby("preassigned_axis")
    .agg(
        n=("subject_id","size"),
        female=("gender",lambda x:x.map(is_female).sum())
    )
)

print("CONFIRM gender")
display(
    confirm.groupby("preassigned_axis")
    .agg(
        n=("subject_id","size"),
        female=("gender",lambda x:x.map(is_female).sum())
    )
)


DEV composition


,preassigned_axis,candidate_source,n
0,COMPETITIVE,V5_WAVE_A,7
1,COMPETITIVE,V5_WIKIDATA_EXPANSION,33
2,PROJECT,V5_WAVE_A,2
3,PROJECT,V5_WIKIDATA_EXPANSION,48
4,STATUS,V5_WAVE_A,6
5,STATUS,V5_WIKIDATA_EXPANSION,64


CONFIRM composition


,preassigned_axis,candidate_source,n
0,COMPETITIVE,V5_WAVE_A,3
1,COMPETITIVE,V5_WIKIDATA_EXPANSION,17
2,PROJECT,V5_WAVE_A,2
3,PROJECT,V5_WIKIDATA_EXPANSION,23
4,STATUS,V5_WAVE_A,2
5,STATUS,V5_WIKIDATA_EXPANSION,33


DEV gender


,n,female
preassigned_axis,,
COMPETITIVE,40,8
PROJECT,50,13
STATUS,70,14


CONFIRM gender


,n,female
preassigned_axis,,
COMPETITIVE,20,4
PROJECT,25,11
STATUS,35,7


## 7. Final freeze + DEV-only event intake

In [9]:

dev_path = ART / "V5_DEV_SUBJECT_ROSTER_160.csv"
confirm_path = ART / "V5_CONFIRM_SUBJECT_ROSTER_80_SEALED.csv"

dev.to_csv(dev_path, index=False)
confirm.to_csv(confirm_path, index=False)

event_template = dev[
    ["subject_id","name","preassigned_axis"]
].copy()

for c in [
    "event_year","polarity","event_type","event_description",
    "source_url","source_title","source_publisher","source_date",
    "source_quality","eligibility_note","exclude","exclude_reason"
]:
    event_template[c] = ""

event_path = ART / "V5_DEV_EVENT_INTAKE_TEMPLATE.csv"
event_template.to_csv(event_path, index=False)

query_cache_hashes = {
    p.name: sha256_file(p)
    for p in sorted(QUERY_CACHE.glob("*.json"))
}

freeze = {
    "version":"V5_FINAL_ROSTER_FREEZE_V1",
    "notebook_version":NOTEBOOK_VERSION,
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":"V5_FINAL_DEV_AND_CONFIRM_ROSTERS_FROZEN_READY_FOR_DEV_EVENT_COLLECTION",
    "dev_n":160,
    "confirm_n":80,
    "dev_axis_counts":dev.preassigned_axis.value_counts().to_dict(),
    "confirm_axis_counts":confirm.preassigned_axis.value_counts().to_dict(),
    "dev_roster_sha256":sha256_file(dev_path),
    "confirm_roster_sha256":sha256_file(confirm_path),
    "event_template_sha256":sha256_file(event_path),
    "wave_a_sha256":sha256_file(WAVE_A_PATH),
    "expansion_request_sha256":sha256_file(EXP_REQ_PATH),
    "birth_snapshot_sha256":sha256_file(BIRTH_PATH),
    "original_seed_sha256":sha256_file(ORIGINAL_SEED_PATH),
    "role_query_snapshot_sha256":sha256_file(role_snapshot_path),
    "strict_aa_expansion_pool_sha256":sha256_file(resolved_new_path),
    "spec_sha256":sha256_file(SPEC_PATH),
    "wikidata_query_cache_sha256":query_cache_hashes,
    "selection_seed":SELECTION_SEED,
    "membership_used_event_outcomes":False,
    "membership_used_chronology":False,
    "membership_used_pairability":False,
    "membership_used_astrology":False,
    "membership_used_control":False,
    "event_collection": {
        "DEV_allowed":True,
        "CONFIRM_allowed":False
    },
    "astrology_generation_allowed":False,
    "next_rule":(
        "Collect source-verifiable events for V5 DEV only. "
        "Freeze all DEV events before pairing. "
        "Do not inspect or collect CONFIRM events. "
        "Do not generate astrology until the pre-astrology local-pair/nuisance quality gate passes."
    )
}

freeze_path = ART / "V5_FINAL_ROSTER_FREEZE_DECISION.json"
json.dump(
    freeze,
    open(freeze_path,"w",encoding="utf-8"),
    ensure_ascii=False,
    indent=2
)

print(json.dumps(freeze,ensure_ascii=False,indent=2))


{
  "version": "V5_FINAL_ROSTER_FREEZE_V1",
  "notebook_version": "SAJU_ML_V5_ROLE_ONLY_EXPANSION_FINAL_FREEZE_20260817",
  "created_at": "2026-08-17T02:30:41",
  "status": "V5_FINAL_DEV_AND_CONFIRM_ROSTERS_FROZEN_READY_FOR_DEV_EVENT_COLLECTION",
  "dev_n": 160,
  "confirm_n": 80,
  "dev_axis_counts": {
    "STATUS": 70,
    "PROJECT": 50,
    "COMPETITIVE": 40
  },
  "confirm_axis_counts": {
    "STATUS": 35,
    "PROJECT": 25,
    "COMPETITIVE": 20
  },
  "dev_roster_sha256": "a7c5a8cbb9855f952d7fbbde7ed6c358d0053dfbec8b2b09f66430413f8a1316",
  "confirm_roster_sha256": "3144f52a8e9304c5cf1773a1dc346b58a1e41fa6af784b3a4ea64a6d318dbfac",
  "event_template_sha256": "9cc25d854bc92678e78494abc9d5b228140b3de232e8a64e5f347b81ed1dac9a",
  "wave_a_sha256": "35ac1a96c66b89299ac5140f45ec11c0ca2cb8088ee5efcca3fd0c0264a08b69",
  "expansion_request_sha256": "318d3e3e5e681d1d3091fd03c0fdde4d1333a79e20ae1a7aca73f581ef0963c0",
  "birth_snapshot_sha256": "ca28a3fea1250b2ea01eb06a54b4921ec0bf790fdc14c5

## Send back exactly these 3 files

```text
V5_FINAL_ROSTER_FREEZE_DECISION.json
V5_DEV_SUBJECT_ROSTER_160.csv
V5_DEV_EVENT_INTAKE_TEMPLATE.csv
```

Do **not** send the CONFIRM roster unless explicitly requested later.

After this status:

`V5_FINAL_DEV_AND_CONFIRM_ROSTERS_FROZEN_READY_FOR_DEV_EVENT_COLLECTION`

the next step is DEV event collection only.
